# Bioresearch Drug Discovery Lab — GRPO Training (Colab)

End-to-end Colab notebook that trains a small instruct model with **GRPO (Group Relative Policy Optimization)** against the live Bioresearch OpenEnv server, using the long-horizon lab tasks (`protein_hypothesis_lab`, `target_discovery_lab`, `curriculum_self_play`).

**What this notebook does**

1. Installs Unsloth + TRL + the Bioresearch repo.
2. Starts the OpenEnv server in the background (inside the Colab VM).
3. Defines a **dense reward function** that rolls out an agent episode against the server and returns the terminal + per-step aggregated reward.
4. Loads `Qwen/Qwen2.5-1.5B-Instruct` with Unsloth 4-bit + LoRA.
5. Runs a short GRPO training loop (~150 steps ≈ 45 min on a free T4).
6. Plots the reward curve before vs. after training — the headline hackathon deliverable.

> The server is **deterministic** (tools return fixed data), so every `(prompt, response)` pair yields the same reward — exactly what GRPO needs.

## 1 · Setup

Install dependencies and clone the Bioresearch environment repo. The first cell assumes a fresh Colab runtime with a GPU attached.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.26" "trl<0.11" peft accelerate bitsandbytes
!pip install -q httpx fastapi uvicorn pydantic pandas matplotlib

import os, subprocess, sys
if not os.path.isdir("bioresearch"):
    subprocess.run(["git", "clone", "https://huggingface.co/spaces/YOUR_HF_USERNAME/bioresearch", "bioresearch"], check=False)
    if not os.path.isdir("bioresearch"):
        # Fallback: clone from an arbitrary working path — user can swap for their own remote.
        print("⚠️  Replace YOUR_HF_USERNAME in the clone URL, or upload the repo manually to /content/bioresearch")
sys.path.insert(0, "/content/bioresearch")
print("✅ Setup complete")

## 2 · Start the OpenEnv server

We launch `uvicorn` in a background subprocess so the reward function can hit `http://127.0.0.1:8000` during training.

In [ ]:
import subprocess, time, httpx

server_proc = subprocess.Popen(
    ["uvicorn", "server.app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/bioresearch",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

# Wait for the server to accept connections.
for _ in range(40):
    try:
        r = httpx.get("http://127.0.0.1:8000/info", timeout=1.0)
        if r.status_code == 200:
            print("✅ Server is up:", r.json())
            break
    except Exception:
        time.sleep(1.0)
else:
    raise RuntimeError("OpenEnv server failed to start")

## 3 · Reward function — rollout a full lab episode

For every completion the policy produces we:

1. Parse it as a JSON tool-call or submission.
2. Replay up to `MAX_STEPS` steps against the OpenEnv server (same `task_id` every rollout — deterministic).
3. Return the **aggregated reward** (per-step process rewards + terminal reward).

This is the "dense signal" GRPO needs. A single-turn reward function would work too but collapses to the classifier we already have.

In [ ]:
import json, re
from client import BioresearchEnv
from models import BioresearchAction

SERVER_URL = "http://127.0.0.1:8000"
TASK_TYPE = "protein_hypothesis_lab"   # swap for target_discovery_lab / curriculum_self_play
MAX_STEPS = 6                           # keep short for training throughput

env_client = BioresearchEnv(base_url=SERVER_URL)

JSON_BLOCK_RE = re.compile(r"\{[\s\S]*\}")

def parse_completion(text: str):
    """Extract the first JSON object from the model output."""
    m = JSON_BLOCK_RE.search(text or "")
    if not m:
        return {"submit": True, "answer": "", "reasoning": text[:400]}
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return {"submit": True, "answer": "", "reasoning": text[:400]}


def reward_lab_episode(prompts, completions, **kwargs) -> list[float]:
    """TRL GRPO reward function. One reward per completion.

    Each completion is treated as a single agent turn; we then let the
    environment auto-submit on the final step to guarantee a terminal reward.
    """
    rewards = []
    for comp in completions:
        text = comp if isinstance(comp, str) else comp[0]["content"]
        parsed = parse_completion(text)

        obs = env_client.reset(task_type=TASK_TYPE)
        total = 0.0
        for step_i in range(MAX_STEPS):
            is_last = step_i == MAX_STEPS - 1
            action = BioresearchAction(
                task_id=obs.task_id,
                tool_name=parsed.get("tool") if not is_last else None,
                tool_args=parsed.get("args") if not is_last else None,
                submit=bool(parsed.get("submit")) or is_last,
                answer=parsed.get("answer", ""),
                reasoning=parsed.get("reasoning", text[:400]),
                go_terms=parsed.get("go_terms"),
                proposed_intervention=parsed.get("intervention"),
            )
            obs = env_client.step(action)
            total += obs.reward or 0.0
            if obs.done:
                break
        rewards.append(float(total))
    return rewards


# Sanity-check on a trivial completion.
print("Sanity reward:", reward_lab_episode(["dummy"], ['{"submit": true, "answer": "unknown"}']))

## 4 · Build the training prompt dataset

Each training "prompt" is just an opening brief for `protein_hypothesis_lab`. The policy produces a JSON tool-call or submission per completion, and GRPO ranks them against each other.

In [ ]:
from datasets import Dataset

SYSTEM = (
    "You are a biomedical research agent. Respond with a SINGLE JSON object and nothing else.\n"
    "Call a tool with: {\"tool\": \"get_ppi\", \"args\": {\"gene\": \"TP53\"}}\n"
    "Or submit with: {\"submit\": true, \"answer\": \"<disease>\", \"reasoning\": \"...\", "
    "\"go_terms\": [\"GO:xxxxxxx\"], \"intervention\": {\"mode\": \"inhibit\", \"target\": \"TP53\"}}\n"
    "Tools: get_interpro, get_ppi, get_go, get_sequence, get_pathway, get_subcellular_location, search_catalogue."
)

# Pull 16 distinct opening briefs.
prompts = []
for i in range(16):
    obs = env_client.reset(task_type=TASK_TYPE)
    prompts.append({
        "prompt": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": obs.question[:1800]},
        ],
    })

train_ds = Dataset.from_list(prompts)
print(train_ds)

## 5 · Load the model with Unsloth (4-bit + LoRA)

We use `Qwen2.5-1.5B-Instruct` because it fits on a free T4 and is small enough to show meaningful policy updates in ~45 minutes. Swap for `Qwen2.5-7B-Instruct` on an A100 for the deluxe demo.

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
tokenizer.padding_side = "left"
print("✅ Model loaded")

## 6 · GRPO training loop

We use TRL's `GRPOTrainer` with our custom reward function. `num_generations=4` gives the algorithm a tight group of samples to rank per prompt.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="grpo_bioresearch",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=1024,
    max_completion_length=256,
    max_steps=150,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    beta=0.04,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=train_ds,
    reward_funcs=[reward_lab_episode],
)

print("✅ Trainer ready")

In [ ]:
trainer.train()

## 7 · Plot the reward curve (the hackathon deliverable)

We pull the per-step reward trace from TRL's log history and plot a smoothed curve.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_df = pd.DataFrame(trainer.state.log_history)
reward_col = next((c for c in log_df.columns if "reward" in c and "std" not in c), None)
print("Available columns:", list(log_df.columns))
print("Using reward column:", reward_col)

if reward_col:
    series = log_df[reward_col].dropna().reset_index(drop=True)
    smooth = series.rolling(window=10, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(series, alpha=0.35, label="reward (raw)")
    ax.plot(smooth, color="#1f77b4", linewidth=2.5, label="reward (10-step EMA)")
    ax.set_xlabel("GRPO step")
    ax.set_ylabel("Mean episode reward")
    ax.set_title(f"Bioresearch Drug Discovery Lab — GRPO on {TASK_TYPE}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("grpo_reward_curve.png", dpi=150)
    plt.show()
    print(f"Baseline (first 10 steps): {series.head(10).mean():.4f}")
    print(f"Final    (last 10 steps): {series.tail(10).mean():.4f}")
else:
    print("⚠️  No reward column found — check TRL version/logging setup.")

## 8 · Save + teardown

Save the LoRA adapter and shut the server down cleanly.

In [ ]:
model.save_pretrained("grpo_bioresearch_lora")
tokenizer.save_pretrained("grpo_bioresearch_lora")

server_proc.terminate()
server_proc.wait(timeout=5)
print("✅ LoRA saved + server stopped")